# 08 — Hyperparameter Tuning, Threshold Calibration, and Evaluation

## Objective

Select model configurations using the designated development data, calibrate anomaly-score thresholds without using the final test labels, and evaluate the finalized models on the held-out test data.

This notebook separates three activities:

1. **Hyperparameter tuning** — select model configurations using the designated development data.
2. **Threshold calibration** — determine anomaly-detection thresholds from normal development data.
3. **Final evaluation** — apply the selected models and fixed thresholds to the held-out evaluation data.

The final test/attack labels are not used for hyperparameter selection or threshold calibration.

## Evaluation Principle

An anomaly-detection model produces a numerical score that represents how unusual an observation or sequence is. A threshold converts this score into a binary prediction.

Let $s_i$ denote the anomaly score for observation or sequence $i$, and let $\tau$ denote the selected threshold. The predicted label is

$$
\hat{y}_i =
\begin{cases}
1, & s_i > \tau,\\
0, & s_i \leq \tau,
\end{cases}
$$

where $\hat{y}_i = 1$ indicates a detected anomaly and $\hat{y}_i = 0$ indicates no detected anomaly.

### Reconstruction Error

For an autoencoder, reconstruction error measures the difference between the input and its reconstructed output.

Let $X_i \in \mathbb{R}^{L \times F}$ be input sequence $i$, and let $\hat{X}_i \in \mathbb{R}^{L \times F}$ be its reconstruction, where:

- $L$ is the sequence length;
- $F$ is the number of input features;
- $x_{i,t,f}$ is the observed value at timestep $t$ and feature $f$;
- $\hat{x}_{i,t,f}$ is the corresponding reconstructed value.

The mean squared reconstruction error for sequence $i$ is

$$
s_i =
\frac{1}{L F}
\sum_{t=1}^{L}
\sum_{f=1}^{F}
\left( x_{i,t,f} - \hat{x}_{i,t,f} \right)^2 .
$$

A larger value indicates a greater reconstruction discrepancy. However, a large reconstruction error is not, by itself, proof of an attack.

### Important Implementation Distinction

The equation above defines a sequence-level reconstruction score. It does not establish that every model in this project uses the same scoring procedure.

Before evaluating the models, the implementation must specify whether each method produces scores per observation, per timestep, or per sequence. For sliding windows, the relationship between sequence scores and the original timestamps must also be defined to ensure that predictions and ground-truth labels are aligned correctly.

Thresholds must be calibrated using the designated normal development data and fixed before final evaluation. Final test labels must not be used to select thresholds or tune model configurations.

## Data Separation

The project preserves the chronological structure of the SWaT time series.

Data used for:

- model training,
- hyperparameter selection,
- threshold calibration,
- and final evaluation

must remain explicitly separated.

In particular, final evaluation labels are not used to select hyperparameters or thresholds. This prevents information from the final evaluation set from influencing model decisions.

## Model Scope

The evaluation covers the anomaly-detection methods defined for this project:

### Baselines

- Rule-Based Detection
- Rolling Statistical Detection
- Simple Statistical Threshold

### Traditional Machine Learning

- Isolation Forest
- One-Class SVM
- Local Outlier Factor in novelty-detection mode

### Deep Learning

- LSTM Autoencoder
- TCN Autoencoder
- USAD

The preliminary TCN and USAD training results from Notebook 07 are treated as experimental evidence, not as final model-selection results.

## Evaluation Metrics

Performance will be assessed using appropriate classification and detection metrics, including:

- Precision
- Recall
- F1-score
- Confusion matrix
- False-positive and false-negative counts

Where appropriate, event-level behavior will also be examined in the subsequent error and event analysis stage.

No single metric will be interpreted in isolation.

In [ ]:
from pathlib import Path

# Locate the project root using the existing project structure.
CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (directory / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected a directory containing 'data/raw'. "
        "Run this notebook from within the SWaT project directory."
    )

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

print(f"Raw data directory exists: {DATA_RAW_DIR.is_dir()}")
print(f"Notebooks directory exists: {NOTEBOOKS_DIR.is_dir()}")

Raw data directory exists: True
Notebooks directory exists: True


In [7]:
# Verify model artifacts in their actual project locations.

MODEL_DIR = PROJECT_ROOT / "models"

EXPECTED_ARTIFACTS = {
    "Isolation Forest": [
        MODEL_DIR / "isolation_forest.joblib",
    ],
    "One-Class SVM": [
        MODEL_DIR / "sgd_one_class_svm.joblib",
    ],
    "Local Outlier Factor": [
        MODEL_DIR / "lof_model.joblib",
    ],
    "LSTM Autoencoder": [
        MODEL_DIR / "lstm_autoencoder.pth",
    ],
    "TCN Autoencoder": [
        MODEL_DIR / "tcn_preliminary" / "tcn_autoencoder_best.pth",
    ],
    "USAD": [
        MODEL_DIR / "usad_preliminary" / "usad_best.pth",
    ],
    "Preprocessing": [
        MODEL_DIR / "scaler.joblib",
    ],
    "Evaluation sequences": [
        MODEL_DIR / "evaluation_sequences.npz",
    ],
}

print("NOTEBOOK 08 — MODEL ARTIFACT VERIFICATION")
print("=" * 48)

for model_name, paths in EXPECTED_ARTIFACTS.items():
    print(f"\n{model_name}:")

    for path in paths:
        if path.is_file():
            print(f"  [FOUND] {path.relative_to(PROJECT_ROOT)}")
        else:
            print(f"  [NOT FOUND] {path.relative_to(PROJECT_ROOT)}")

NOTEBOOK 08 — MODEL ARTIFACT VERIFICATION

Isolation Forest:
  [FOUND] models\isolation_forest.joblib

One-Class SVM:
  [FOUND] models\sgd_one_class_svm.joblib

Local Outlier Factor:
  [FOUND] models\lof_model.joblib

LSTM Autoencoder:
  [FOUND] models\lstm_autoencoder.pth

TCN Autoencoder:
  [FOUND] models\tcn_preliminary\tcn_autoencoder_best.pth

USAD:
  [FOUND] models\usad_preliminary\usad_best.pth

Preprocessing:
  [FOUND] models\scaler.joblib

Evaluation sequences:
  [FOUND] models\evaluation_sequences.npz


In [9]:
# Load the existing evaluation arrays once for use in Notebook 08.

import numpy as np

EVALUATION_FILE = PROJECT_ROOT / "models" / "evaluation_sequences.npz"

if not EVALUATION_FILE.is_file():
    raise FileNotFoundError(
        f"Evaluation file not found: {EVALUATION_FILE}"
    )

with np.load(EVALUATION_FILE, allow_pickle=False) as archive:
    EVALUATION_ARRAYS = {
        name: archive[name]
        for name in archive.files
    }

print("EVALUATION DATA READY")
print("=" * 40)

for name, array in EVALUATION_ARRAYS.items():
    print(
        f"{name}: "
        f"shape={array.shape}, "
        f"dtype={array.dtype}"
    )

EVALUATION DATA READY
X_threshold_seq: shape=(47408, 30, 42), dtype=float32
X_validation_seq: shape=(79031, 30, 42), dtype=float32
threshold_end_indices: shape=(47408,), dtype=int64
validation_end_indices: shape=(79031,), dtype=int64


In [10]:
# Prepare the existing evaluation arrays for subsequent model scoring.

X_threshold_seq = EVALUATION_ARRAYS["X_threshold_seq"]
X_validation_seq = EVALUATION_ARRAYS["X_validation_seq"]

threshold_end_indices = EVALUATION_ARRAYS["threshold_end_indices"]
validation_end_indices = EVALUATION_ARRAYS["validation_end_indices"]

# Validate sequence dimensions and index alignment.
assert X_threshold_seq.ndim == 3
assert X_validation_seq.ndim == 3

assert X_threshold_seq.shape[1:] == (30, 42)
assert X_validation_seq.shape[1:] == (30, 42)

assert len(X_threshold_seq) == len(threshold_end_indices)
assert len(X_validation_seq) == len(validation_end_indices)

print("Evaluation arrays prepared successfully.")
print(f"Threshold sequences: {X_threshold_seq.shape}")
print(f"Validation sequences: {X_validation_seq.shape}")
print(f"Threshold index range: {threshold_end_indices.min()}–{threshold_end_indices.max()}")
print(f"Validation index range: {validation_end_indices.min()}–{validation_end_indices.max()}")

Evaluation arrays prepared successfully.
Threshold sequences: (47408, 30, 42)
Validation sequences: (79031, 30, 42)
Threshold index range: 29–47436
Validation index range: 29–79059


In [14]:
import joblib
import numpy as np
import pandas as pd


import warnings
from sklearn.exceptions import InconsistentVersionWarning

warnings.filterwarnings(
    "ignore",
    category=InconsistentVersionWarning,
)

# Load the existing fitted models. Do not retrain them.
MODEL_DIR = PROJECT_ROOT / "models"

scaler = joblib.load(MODEL_DIR / "scaler.joblib")

isolation_forest = joblib.load(
    MODEL_DIR / "isolation_forest.joblib"
)

ocsvm = joblib.load(
    MODEL_DIR / "sgd_one_class_svm.joblib"
)

lof = joblib.load(
    MODEL_DIR / "lof_model.joblib"
)

# Use the feature order recorded by the fitted scaler.
FEATURE_COLUMNS = list(scaler.feature_names_in_)

if len(FEATURE_COLUMNS) != 42:
    raise ValueError(
        f"Expected 42 features, found {len(FEATURE_COLUMNS)}."
    )

print("Traditional ML artifacts loaded successfully.")
print(f"Feature count: {len(FEATURE_COLUMNS)}")

Traditional ML artifacts loaded successfully.
Feature count: 42


In [17]:
# Define the normal dataset path within the project.

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
NORMAL_PATH = DATA_RAW_DIR / "normal.csv"

if not NORMAL_PATH.is_file():
    raise FileNotFoundError(
        f"Normal dataset not found: {NORMAL_PATH}"
    )

print("Normal dataset path verified.")
# print(f"Path: {NORMAL_PATH}")

Normal dataset path verified.


In [19]:
# Reconstruct the established normal calibration and validation partitions.
import warnings

# Suppress pandas UserWarnings
warnings.filterwarnings('ignore', category=UserWarning)

TRAIN_ROWS = 221_366
MONITOR_ROWS = 47_435
THRESHOLD_ROWS = 47_437
VALIDATION_ROWS = 79_060

EXCLUDED_FEATURES = [
    "P202", "P204", "P206", "P301", "P401",
    "P404", "P502", "P601", "P603",
]

ALL_SENSOR_COLUMNS = FEATURE_COLUMNS + EXCLUDED_FEATURES

# Read the source data and reproduce Notebook 06's eligibility filter.
normal_df = pd.read_csv(NORMAL_PATH, low_memory=False)
normal_df.columns = normal_df.columns.str.strip()

missing_columns = [
    col for col in ["Timestamp", *ALL_SENSOR_COLUMNS]
    if col not in normal_df.columns
]

if missing_columns:
    raise KeyError(f"Required columns missing: {missing_columns}")

normal_df["Timestamp"] = pd.to_datetime(
    normal_df["Timestamp"],
    errors="coerce",
)

eligible_mask = (
    normal_df[ALL_SENSOR_COLUMNS].notna().all(axis=1)
    & normal_df["Timestamp"].notna()
)

eligible_normal = normal_df.loc[eligible_mask].copy()

# Verify the established partition sizes before extracting rows.
expected_rows = (
    TRAIN_ROWS + MONITOR_ROWS + THRESHOLD_ROWS + VALIDATION_ROWS
)

if len(eligible_normal) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows:,} eligible observations, "
        f"but found {len(eligible_normal):,}."
    )

threshold_start = TRAIN_ROWS + MONITOR_ROWS
threshold_end = threshold_start + THRESHOLD_ROWS

threshold_df = eligible_normal.iloc[
    threshold_start:threshold_end
].copy()

validation_df = eligible_normal.iloc[threshold_end:].copy()

# Verify partition sizes and chronology.
assert len(threshold_df) == THRESHOLD_ROWS
assert len(validation_df) == VALIDATION_ROWS

assert threshold_df["Timestamp"].is_monotonic_increasing
assert validation_df["Timestamp"].is_monotonic_increasing

assert (
    threshold_df["Timestamp"].iloc[-1]
    < validation_df["Timestamp"].iloc[0]
)

print("NORMAL PARTITIONS RECONSTRUCTED")
print("=" * 48)
print(f"Eligible observations: {len(eligible_normal):,}")
print(f"Threshold observations: {len(threshold_df):,}")
print(f"Validation observations: {len(validation_df):,}")
print(
    "Threshold period:",
    threshold_df["Timestamp"].iloc[0],
    "to",
    threshold_df["Timestamp"].iloc[-1],
)
print(
    "Validation period:",
    validation_df["Timestamp"].iloc[0],
    "to",
    validation_df["Timestamp"].iloc[-1],
)
print("Chronological separation verified: True")

NORMAL PARTITIONS RECONSTRUCTED
Eligible observations: 395,298
Threshold observations: 47,437
Validation observations: 79,060
Threshold period: 2016-01-01 02:23:18 to 2016-01-01 15:51:19
Validation period: 2016-01-01 15:51:20 to 2016-01-02 14:59:59
Chronological separation verified: True


In [20]:
# Generate anomaly scores using the saved traditional ML models.
# Higher scores indicate greater anomalousness.

# Extract the established 42 features in the saved scaler's order.
X_threshold_raw = threshold_df[FEATURE_COLUMNS]
X_validation_raw = validation_df[FEATURE_COLUMNS]

# Apply the existing scaler; do not fit it again.
X_threshold = scaler.transform(X_threshold_raw).astype(np.float32)
X_validation = scaler.transform(X_validation_raw).astype(np.float32)

# Verify the expected dimensions and numerical validity.
assert X_threshold.shape == (THRESHOLD_ROWS, 42)
assert X_validation.shape == (VALIDATION_ROWS, 42)

assert np.isfinite(X_threshold).all()
assert np.isfinite(X_validation).all()

# 1. Isolation Forest
if_threshold_scores = -isolation_forest.score_samples(X_threshold)
if_validation_scores = -isolation_forest.score_samples(X_validation)

# 2. One-Class SVM
ocsvm_threshold_scores = -ocsvm.score_samples(X_threshold)
ocsvm_validation_scores = -ocsvm.score_samples(X_validation)

# 3. Local Outlier Factor (novelty mode)
lof_threshold_scores = -lof.score_samples(X_threshold)
lof_validation_scores = -lof.score_samples(X_validation)

# Verify score dimensions and numerical validity.
score_arrays = {
    "IF threshold": if_threshold_scores,
    "IF validation": if_validation_scores,
    "OCSVM threshold": ocsvm_threshold_scores,
    "OCSVM validation": ocsvm_validation_scores,
    "LOF threshold": lof_threshold_scores,
    "LOF validation": lof_validation_scores,
}

for name, scores in score_arrays.items():
    expected_length = (
        THRESHOLD_ROWS if "threshold" in name else VALIDATION_ROWS
    )

    if scores.shape != (expected_length,):
        raise ValueError(
            f"{name}: expected {(expected_length,)}, got {scores.shape}."
        )

    if not np.isfinite(scores).all():
        raise ValueError(f"{name} contains NaN or infinite values.")

print("TRADITIONAL ML SCORING COMPLETED")
print("=" * 60)

for name, scores in score_arrays.items():
    print(
        f"{name:20s} | "
        f"n={len(scores):,} | "
        f"min={scores.min():.6f} | "
        f"median={np.median(scores):.6f} | "
        f"max={scores.max():.6f}"
    )

print("\nAll six score arrays passed dimension and finiteness checks.")

TRADITIONAL ML SCORING COMPLETED
IF threshold         | n=47,437 | min=0.437794 | median=0.497420 | max=0.676735
IF validation        | n=79,060 | min=0.442309 | median=0.507095 | max=0.679314
OCSVM threshold      | n=47,437 | min=-15.859379 | median=-15.356825 | max=-13.448579
OCSVM validation     | n=79,060 | min=-16.083435 | median=-15.271841 | max=-9.913305
LOF threshold        | n=47,437 | min=1.008754 | median=10.446300 | max=59.167336
LOF validation       | n=79,060 | min=1.139901 | median=12.727806 | max=50.844120

All six score arrays passed dimension and finiteness checks.


In [22]:
# Persist traditional ML anomaly scores for Notebook 08 evaluation.

SCORES_FILE = MODEL_DIR / "traditional_ml_scores.npz"

np.savez_compressed(
    SCORES_FILE,
    if_threshold_scores=if_threshold_scores,
    if_validation_scores=if_validation_scores,
    ocsvm_threshold_scores=ocsvm_threshold_scores,
    ocsvm_validation_scores=ocsvm_validation_scores,
    lof_threshold_scores=lof_threshold_scores,
    lof_validation_scores=lof_validation_scores,
)

if not SCORES_FILE.is_file():
    raise FileNotFoundError(
        f"Traditional ML score archive was not created: {SCORES_FILE}"
    )

print("TRADITIONAL ML SCORES SAVED")
print("=" * 60)
# print(f"Path: {SCORES_FILE}")
print(f"File size: {SCORES_FILE.stat().st_size / (1024**2):.2f} MB")
print("Saved arrays:")
print("  - IF threshold")
print("  - IF validation")
print("  - OCSVM threshold")
print("  - OCSVM validation")
print("  - LOF threshold")
print("  - LOF validation")

TRADITIONAL ML SCORES SAVED
File size: 1.63 MB
Saved arrays:
  - IF threshold
  - IF validation
  - OCSVM threshold
  - OCSVM validation
  - LOF threshold
  - LOF validation


In [23]:
# Align traditional ML scores to the endpoint rows used by sequence models.

if_threshold_seq_scores = if_threshold_scores[threshold_end_indices]
if_validation_seq_scores = if_validation_scores[validation_end_indices]

ocsvm_threshold_seq_scores = ocsvm_threshold_scores[threshold_end_indices]
ocsvm_validation_seq_scores = ocsvm_validation_scores[validation_end_indices]

lof_threshold_seq_scores = lof_threshold_scores[threshold_end_indices]
lof_validation_seq_scores = lof_validation_scores[validation_end_indices]

# Verify alignment with the existing sequence arrays.
expected_threshold_sequences = len(X_threshold_seq)
expected_validation_sequences = len(X_validation_seq)

aligned_scores = {
    "IF threshold": if_threshold_seq_scores,
    "IF validation": if_validation_seq_scores,
    "OCSVM threshold": ocsvm_threshold_seq_scores,
    "OCSVM validation": ocsvm_validation_seq_scores,
    "LOF threshold": lof_threshold_seq_scores,
    "LOF validation": lof_validation_seq_scores,
}

for name, scores in aligned_scores.items():
    expected_length = (
        expected_threshold_sequences
        if "threshold" in name
        else expected_validation_sequences
    )

    if scores.shape != (expected_length,):
        raise ValueError(
            f"{name}: expected {(expected_length,)}, got {scores.shape}."
        )

    if not np.isfinite(scores).all():
        raise ValueError(f"{name} contains NaN or infinite values.")

print("TRADITIONAL ML SEQUENCE ALIGNMENT COMPLETED")
print("=" * 60)
print(f"Threshold sequence support:  {expected_threshold_sequences:,}")
print(f"Validation sequence support: {expected_validation_sequences:,}")
print()
print("All six aligned score arrays passed dimension and finiteness checks.")

TRADITIONAL ML SEQUENCE ALIGNMENT COMPLETED
Threshold sequence support:  47,408
Validation sequence support: 79,031

All six aligned score arrays passed dimension and finiteness checks.


In [25]:
# Score the saved LSTM Autoencoder on normal calibration and validation sequences.

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# 1. Reconstruct the exact architecture used in Notebook 05.

class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        bottleneck_size,
        dropout,
    ):
        super().__init__()

        self.encoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.encoder_projection = nn.Linear(
            hidden_size,
            bottleneck_size,
        )

        self.decoder_projection = nn.Linear(
            bottleneck_size,
            hidden_size,
        )

        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.output_layer = nn.Linear(
            hidden_size,
            input_size,
        )

    def forward(self, x):
        _, (hidden, _) = self.encoder(x)

        latent = F.leaky_relu(
            self.encoder_projection(hidden[-1]),
            negative_slope=0.01,
        )

        decoder_input = F.leaky_relu(
            self.decoder_projection(latent),
            negative_slope=0.01,
        )

        decoder_input = decoder_input.unsqueeze(1).repeat(
            1,
            x.size(1),
            1,
        )

        decoded, _ = self.decoder(decoder_input)

        return self.output_layer(decoded)


# 2. Select the available inference device.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. Load the saved model checkpoint.
LSTM_CHECKPOINT = MODEL_DIR / "lstm_autoencoder.pth"

if not LSTM_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"LSTM checkpoint not found: {LSTM_CHECKPOINT}"
    )

lstm_model = LSTMAutoencoder(
    input_size=42,
    hidden_size=64,
    num_layers=2,
    bottleneck_size=16,
    dropout=0.2,
).to(DEVICE)

try:
    lstm_state = torch.load(
        LSTM_CHECKPOINT,
        map_location=DEVICE,
        weights_only=True,
    )
except TypeError:
    lstm_state = torch.load(
        LSTM_CHECKPOINT,
        map_location=DEVICE,
    )

lstm_model.load_state_dict(lstm_state)
lstm_model.eval()


# 4. Calculate one MSE reconstruction score per sequence.
def calculate_reconstruction_errors(
    model,
    sequences,
    device,
    batch_size=512,
):
    model.eval()
    scores = []

    with torch.no_grad():
        for start in range(0, len(sequences), batch_size):
            batch = torch.as_tensor(
                np.asarray(
                    sequences[start:start + batch_size],
                    dtype=np.float32,
                ),
                dtype=torch.float32,
                device=device,
            )

            reconstruction = model(batch)

            batch_scores = (
                (reconstruction - batch) ** 2
            ).mean(dim=(1, 2))

            scores.append(batch_scores.cpu().numpy())

    return np.concatenate(scores).astype(np.float64)


# Confirm input dimensions before scoring.
assert X_threshold_seq.ndim == 3
assert X_validation_seq.ndim == 3
assert X_threshold_seq.shape[1:] == (30, 42)
assert X_validation_seq.shape[1:] == (30, 42)

lstm_threshold_scores = calculate_reconstruction_errors(
    model=lstm_model,
    sequences=X_threshold_seq,
    device=DEVICE,
)

lstm_validation_scores = calculate_reconstruction_errors(
    model=lstm_model,
    sequences=X_validation_seq,
    device=DEVICE,
)


# 5. Verify the outputs.
assert lstm_threshold_scores.shape == (len(X_threshold_seq),)
assert lstm_validation_scores.shape == (len(X_validation_seq),)
assert np.isfinite(lstm_threshold_scores).all()
assert np.isfinite(lstm_validation_scores).all()
assert (lstm_threshold_scores >= 0).all()
assert (lstm_validation_scores >= 0).all()


# 6. Save the LSTM scores for reproducible evaluation.
LSTM_SCORES_FILE = MODEL_DIR / "lstm_autoencoder_scores.npz"

np.savez_compressed(
    LSTM_SCORES_FILE,
    threshold_scores=lstm_threshold_scores,
    validation_scores=lstm_validation_scores,
    threshold_end_indices=threshold_end_indices,
    validation_end_indices=validation_end_indices,
)

print("LSTM AUTOENCODER SCORING COMPLETED")
print("=" * 60)
print(f"Device: {DEVICE}")
print(f"Threshold sequences:  {len(lstm_threshold_scores):,}")
print(f"Validation sequences: {len(lstm_validation_scores):,}")
print()
print("Threshold-calibration score distribution")
print(f"  Minimum: {lstm_threshold_scores.min():.8f}")
print(f"  Mean:    {lstm_threshold_scores.mean():.8f}")
print(f"  Median:  {np.median(lstm_threshold_scores):.8f}")
print(f"  95th pct: {np.percentile(lstm_threshold_scores, 95):.8f}")
print(f"  99th pct: {np.percentile(lstm_threshold_scores, 99):.8f}")
print(f"  Maximum: {lstm_threshold_scores.max():.8f}")
print()
print("Normal validation score distribution")
print(f"  Minimum: {lstm_validation_scores.min():.8f}")
print(f"  Mean:    {lstm_validation_scores.mean():.8f}")
print(f"  Median:  {np.median(lstm_validation_scores):.8f}")
print(f"  95th pct: {np.percentile(lstm_validation_scores, 95):.8f}")
print(f"  99th pct: {np.percentile(lstm_validation_scores, 99):.8f}")
print(f"  Maximum: {lstm_validation_scores.max():.8f}")
print()
# print(f"Scores saved to: {LSTM_SCORES_FILE}")
print("All score arrays passed validation.")

LSTM AUTOENCODER SCORING COMPLETED
Device: cpu
Threshold sequences:  47,408
Validation sequences: 79,031

Threshold-calibration score distribution
  Minimum: 0.00163360
  Mean:    0.00446213
  Median:  0.00324544
  95th pct: 0.01076056
  99th pct: 0.01882628
  Maximum: 0.15577796

Normal validation score distribution
  Minimum: 0.00311918
  Mean:    0.00686866
  Median:  0.00516796
  95th pct: 0.01583000
  99th pct: 0.02630821
  Maximum: 0.34991333

All score arrays passed validation.


In [27]:
# Score the saved TCN Autoencoder on normal calibration and validation sequences.
# 1. Reconstruct the exact TCN architecture from Notebook 07.

class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0,
        )

    def forward(self, x):
        return self.conv(F.pad(x, (self.left_padding, 0)))


class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()

        self.conv1 = CausalConv1d(
            in_channels, out_channels, kernel_size, dilation
        )
        self.conv2 = CausalConv1d(
            out_channels, out_channels, kernel_size, dilation
        )

        self.activation = nn.ReLU()
        self.residual = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv1d(in_channels, out_channels, kernel_size=1)
        )

    def forward(self, x):
        residual = self.residual(x)
        x = self.activation(self.conv1(x))
        x = self.activation(self.conv2(x))
        return self.activation(x + residual)


class TCNAutoencoder(nn.Module):
    def __init__(self, n_features, channels, kernel_size, dilations):
        super().__init__()

        if len(channels) != len(dilations):
            raise ValueError("Each TCN block must have a dilation value.")

        encoder_layers = []
        in_channels = n_features

        for out_channels, dilation in zip(channels, dilations):
            encoder_layers.append(
                TCNBlock(
                    in_channels,
                    out_channels,
                    kernel_size,
                    dilation,
                )
            )
            in_channels = out_channels

        self.encoder = nn.Sequential(*encoder_layers)

        decoder_layers = []
        reversed_channels = list(reversed(channels))
        reversed_dilations = list(reversed(dilations))

        for index, dilation in enumerate(reversed_dilations):
            current_channels = reversed_channels[index]
            next_channels = (
                reversed_channels[index + 1]
                if index + 1 < len(reversed_channels)
                else n_features
            )

            decoder_layers.append(
                CausalConv1d(
                    current_channels,
                    next_channels,
                    kernel_size,
                    dilation,
                )
            )

            if index + 1 < len(reversed_dilations):
                decoder_layers.append(nn.ReLU())

        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError(
                "Expected input shape (batch, timesteps, features)."
            )

        x = x.transpose(1, 2).contiguous()
        x = self.encoder(x)
        x = self.decoder(x)

        return x.transpose(1, 2).contiguous()


# 2. Verify the expected sequence dimensions.
assert X_threshold_seq.shape[1:] == (30, 42)
assert X_validation_seq.shape[1:] == (30, 42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. Load the saved best TCN checkpoint.
TCN_CHECKPOINT = MODEL_DIR / "tcn_preliminary" / "tcn_autoencoder_best.pth"

if not TCN_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"TCN checkpoint not found: {TCN_CHECKPOINT}"
    )

tcn_model = TCNAutoencoder(
    n_features=42,
    channels=[32, 32, 32, 32],
    kernel_size=3,
    dilations=[1, 2, 4, 8],
).to(DEVICE)

try:
    tcn_state = torch.load(
        TCN_CHECKPOINT,
        map_location=DEVICE,
        weights_only=True,
    )
except TypeError:
    tcn_state = torch.load(
        TCN_CHECKPOINT,
        map_location=DEVICE,
    )

tcn_model.load_state_dict(tcn_state)
tcn_model.eval()


# 4. Calculate one mean squared reconstruction error per sequence.
def calculate_tcn_reconstruction_errors(
    model,
    sequences,
    device,
    batch_size=512,
):
    scores = []
    model.eval()

    with torch.no_grad():
        for start in range(0, len(sequences), batch_size):
            batch = torch.as_tensor(
                np.asarray(
                    sequences[start:start + batch_size],
                    dtype=np.float32,
                ),
                dtype=torch.float32,
                device=device,
            )

            reconstruction = model(batch)

            batch_scores = (
                (reconstruction - batch) ** 2
            ).mean(dim=(1, 2))

            scores.append(batch_scores.cpu().numpy())

    return np.concatenate(scores).astype(np.float64)


# 5. Score the normal calibration and validation sequences.
tcn_threshold_scores = calculate_tcn_reconstruction_errors(
    model=tcn_model,
    sequences=X_threshold_seq,
    device=DEVICE,
)

tcn_validation_scores = calculate_tcn_reconstruction_errors(
    model=tcn_model,
    sequences=X_validation_seq,
    device=DEVICE,
)


# 6. Validate the score arrays.
assert tcn_threshold_scores.shape == (len(X_threshold_seq),)
assert tcn_validation_scores.shape == (len(X_validation_seq),)
assert np.isfinite(tcn_threshold_scores).all()
assert np.isfinite(tcn_validation_scores).all()
assert (tcn_threshold_scores >= 0).all()
assert (tcn_validation_scores >= 0).all()


# 7. Save scores with their sequence endpoint indices.
TCN_SCORES_FILE = MODEL_DIR / "tcn_autoencoder_scores.npz"

np.savez_compressed(
    TCN_SCORES_FILE,
    threshold_scores=tcn_threshold_scores,
    validation_scores=tcn_validation_scores,
    threshold_end_indices=threshold_end_indices,
    validation_end_indices=validation_end_indices,
)

# 8. Report the observed score distributions.
print("TCN AUTOENCODER SCORING COMPLETED")
print("=" * 60)
print(f"Device: {DEVICE}")
print(f"Threshold sequences:  {len(tcn_threshold_scores):,}")
print(f"Validation sequences: {len(tcn_validation_scores):,}")
print()

for name, scores in [
    ("Threshold calibration", tcn_threshold_scores),
    ("Normal validation", tcn_validation_scores),
]:
    print(name)
    print(f"  Minimum:  {scores.min():.8f}")
    print(f"  Mean:     {scores.mean():.8f}")
    print(f"  Median:   {np.median(scores):.8f}")
    print(f"  95th pct: {np.percentile(scores, 95):.8f}")
    print(f"  99th pct: {np.percentile(scores, 99):.8f}")
    print(f"  Maximum:  {scores.max():.8f}")
    print()

# print(f"Scores saved to: {TCN_SCORES_FILE}")
print("All score arrays passed validation.")

TCN AUTOENCODER SCORING COMPLETED
Device: cpu
Threshold sequences:  47,408
Validation sequences: 79,031

Threshold calibration
  Minimum:  0.00037321
  Mean:     0.00131288
  Median:   0.00109156
  95th pct: 0.00253684
  99th pct: 0.00390673
  Maximum:  0.00703319

Normal validation
  Minimum:  0.00063398
  Mean:     0.00207530
  Median:   0.00158158
  95th pct: 0.00338929
  99th pct: 0.00588506
  Maximum:  0.14281599

All score arrays passed validation.


In [29]:
# Inspect the attack dataset schema and a small sample.

import pandas as pd

ATTACK_PATH = DATA_RAW_DIR / "attack.csv"

if not ATTACK_PATH.is_file():
    raise FileNotFoundError(f"Attack dataset not found: {ATTACK_PATH}")

attack_sample = pd.read_csv(
    ATTACK_PATH,
    nrows=10,
    low_memory=False
)

print("ATTACK DATASET INSPECTION")
print("=" * 60)
# print(f"File: {ATTACK_PATH}")
print(f"Columns: {len(attack_sample.columns)}")
print("\nColumn names:")
print(attack_sample.columns.tolist())

print("\nData types:")
print(attack_sample.dtypes.to_string())

print("\nFirst five rows:")
display(attack_sample.head())

print("\nMissing values in the sample:")
print(attack_sample.isna().sum().to_string())

ATTACK DATASET INSPECTION
Columns: 53

Column names:
[' Timestamp', 'FIT101', 'LIT101', ' MV101', 'P101', 'P102', ' AIT201', 'AIT202', 'AIT203', 'FIT201', ' MV201', ' P201', ' P202', 'P203', ' P204', 'P205', 'P206', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', ' MV303', 'MV304', 'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P401', 'P402', 'P403', 'P404', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'P502', 'PIT501', 'PIT502', 'PIT503', 'FIT601', 'P601', 'P602', 'P603', 'Normal/Attack']

Data types:
 Timestamp           str
FIT101           float64
LIT101           float64
 MV101             int64
P101               int64
P102               int64
 AIT201          float64
AIT202           float64
AIT203           float64
FIT201             int64
 MV201             int64
 P201              int64
 P202              int64
P203               int64
 P204              int64
P205               int64
P206               int64
DPIT

,Timestamp,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,...,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Normal/Attack
0,28/12/2015 10:29:14 AM,2.493675,817.6742,2,1,1,262.625,8.46533,319.7385,0,...,2,1,249.8718,1.681991,188.7977,0.000128,1,1,1,Attack
1,28/12/2015 10:29:15 AM,2.535951,817.9490,2,1,1,262.625,8.46533,319.7385,0,...,2,1,249.8718,1.681991,188.7977,0.000128,1,1,1,Attack
2,28/12/2015 10:29:16 AM,2.535951,817.9490,2,1,1,262.625,8.46533,319.7385,0,...,2,1,249.8718,1.681991,188.7977,0.000128,1,1,1,Attack
3,28/12/2015 10:29:17 AM,2.569900,818.4592,2,1,1,262.625,8.46533,319.7385,0,...,2,1,249.8718,1.681991,188.7977,0.000128,1,1,1,Attack
4,28/12/2015 10:29:18 AM,2.610575,818.8911,2,1,1,262.625,8.46533,319.7385,0,...,2,1,250.0801,1.681991,188.9740,0.000128,1,1,1,Attack



Missing values in the sample:
 Timestamp       0
FIT101           0
LIT101           0
 MV101           0
P101             0
P102             0
 AIT201          0
AIT202           0
AIT203           0
FIT201           0
 MV201           0
 P201            0
 P202            0
P203             0
 P204            0
P205             0
P206             0
DPIT301          0
FIT301           0
LIT301           0
MV301            0
MV302            0
 MV303           0
MV304            0
P301             0
P302             0
AIT401           0
AIT402           0
FIT401           0
LIT401           0
P401             0
P402             0
P403             0
P404             0
UV401            0
AIT501           0
AIT502           0
AIT503           0
AIT504           0
FIT501           0
FIT502           0
FIT503           0
FIT504           0
P501             0
P502             0
PIT501           0
PIT502           0
PIT503           0
FIT601           0
P601             0
P602             0


In [ ]:
# Validate the attack dataset in chunks before model evaluation.
import csv

CHUNK_SIZE = 50_000

total_rows = 0
invalid_timestamps = 0
missing_timestamps = 0
out_of_order_chunks = 0
missing_values_by_column = None
label_counts = {}
first_timestamp = None
last_timestamp = None
previous_timestamp = None
timestamp_format = "%d/%m/%Y %I:%M:%S %p"

with ATTACK_PATH.open("r", encoding="utf-8-sig", newline="") as file:
    reader = csv.reader(file)
    raw_header = next(reader)

normalized_header = [column.strip() for column in raw_header]

if len(normalized_header) != len(set(normalized_header)):
    raise ValueError("Duplicate column names exist after whitespace normalization.")

required_columns = {"Timestamp", "Normal/Attack", *FEATURE_COLUMNS}
missing_required = required_columns - set(normalized_header)

if missing_required:
    raise ValueError(
        f"Required columns missing from attack.csv: {sorted(missing_required)}"
    )

for chunk in pd.read_csv(
    ATTACK_PATH,
    chunksize=CHUNK_SIZE,
    low_memory=False,
    encoding="utf-8-sig",
):
    chunk.columns = chunk.columns.str.strip()
    chunk_rows = len(chunk)
    total_rows += chunk_rows

    if missing_values_by_column is None:
        missing_values_by_column = pd.Series(0, index=chunk.columns, dtype="int64")

    missing_values_by_column = (
        missing_values_by_column.add(chunk.isna().sum(), fill_value=0)
        .astype("int64")
    )

    labels = chunk["Normal/Attack"].astype("string").str.strip()
    counts = labels.value_counts(dropna=False)

    for label, count in counts.items():
        label_counts[str(label)] = label_counts.get(str(label), 0) + int(count)

    timestamps = pd.to_datetime(
        chunk["Timestamp"].astype("string").str.strip(),
        format=timestamp_format,
        errors="coerce",
    )

    invalid_timestamps += int(timestamps.isna().sum())
    valid_timestamps = timestamps.dropna()

    if len(valid_timestamps) != len(timestamps):
        missing_timestamps += int(timestamps.isna().sum())

    if not valid_timestamps.empty:
        chunk_first = valid_timestamps.iloc[0]
        chunk_last = valid_timestamps.iloc[-1]

        if first_timestamp is None:
            first_timestamp = chunk_first

        if previous_timestamp is not None and chunk_first < previous_timestamp:
            out_of_order_chunks += 1

        if not timestamps.is_monotonic_increasing:
            out_of_order_chunks += 1

        previous_timestamp = chunk_last
        last_timestamp = chunk_last

print("ATTACK DATASET VALIDATION")
print("=" * 60)
print(f"Total rows: {total_rows:,}")
print(f"Total columns: {len(normalized_header)}")
print(f"Invalid timestamps: {invalid_timestamps:,}")
print(f"Missing timestamps: {missing_timestamps:,}")
print(f"Out-of-order checks detected: {out_of_order_chunks}")
print(f"First valid timestamp: {first_timestamp}")
print(f"Last valid timestamp: {last_timestamp}")

print("\nLabel distribution:")
for label, count in sorted(label_counts.items()):
    print(f"  {label}: {count:,}")

print("\nColumns with missing values:")
if missing_values_by_column is None:
    print("No data rows were read.")
else:
    missing_columns = missing_values_by_column[
        missing_values_by_column > 0
    ].sort_values(ascending=False)

    if missing_columns.empty:
        print("None.")
    else:
        print(missing_columns.to_string())

print("\nFeature compatibility:")
print(f"Expected model features: {len(FEATURE_COLUMNS)}")
print(f"All expected features present: {not missing_required}")

ATTACK DATASET VALIDATION
Total rows: 54,621
Total columns: 53
Invalid timestamps: 0
Missing timestamps: 0
Out-of-order checks detected: 0
First valid timestamp: 2015-12-28 10:29:14
Last valid timestamp: 2016-01-02 13:41:11

Label distribution:
  Attack: 54,621

Columns with missing values:
None.

Feature compatibility:
Expected model features: 42
All expected features present: True


In [ ]:
# Prepare attack evaluation sequences.
# Reuse the existing scaler; do not retrain or refit anything.

WINDOW_SIZE = 30

# Load the attack dataset and normalize column names.
attack_df = pd.read_csv(ATTACK_PATH, low_memory=False)
attack_df.columns = attack_df.columns.str.strip()

# Parse timestamps using the verified format.
attack_df["Timestamp"] = pd.to_datetime(
    attack_df["Timestamp"].astype("string").str.strip(),
    format="%d/%m/%Y %I:%M:%S %p",
    errors="raise",
)

# Preserve and validate the ground-truth labels.
attack_labels = attack_df["Normal/Attack"].astype("string").str.strip()

assert attack_labels.eq("Attack").all(), (
    "Unexpected labels found in the attack dataset."
)
assert attack_df["Timestamp"].is_monotonic_increasing, (
    "Attack observations are not in chronological order."
)
assert attack_df["Timestamp"].is_unique, (
    "Duplicate attack timestamps were found."
)
assert all(column in attack_df.columns for column in FEATURE_COLUMNS), (
    "One or more required model features are missing."
)

# Transform using the existing scaler fitted on training data.
X_attack_scaled = scaler.transform(
    attack_df[FEATURE_COLUMNS]
).astype(np.float32)

# Construct independent 30-observation windows.
n_sequences = len(X_attack_scaled) - WINDOW_SIZE + 1

if n_sequences <= 0:
    raise ValueError("Not enough observations to construct attack sequences.")

attack_sequences = np.stack([
    X_attack_scaled[start:start + WINDOW_SIZE]
    for start in range(n_sequences)
]).astype(np.float32)

# Each sequence is associated with its final observation.
attack_end_indices = np.arange(
    WINDOW_SIZE - 1,
    len(attack_df),
    dtype=np.int64,
)

attack_sequence_timestamps = (
    attack_df["Timestamp"]
    .iloc[attack_end_indices]
    .reset_index(drop=True)
)

attack_sequence_labels = (
    attack_labels
    .iloc[attack_end_indices]
    .reset_index(drop=True)
)

# Validate sequence dimensions and label alignment.
assert attack_sequences.shape == (
    54_592,
    WINDOW_SIZE,
    len(FEATURE_COLUMNS),
)
assert len(attack_end_indices) == len(attack_sequences)
assert len(attack_sequence_timestamps) == len(attack_sequences)
assert len(attack_sequence_labels) == len(attack_sequences)
assert np.isfinite(attack_sequences).all()
assert attack_sequence_labels.eq("Attack").all()

# Save the prepared sequences and endpoint indices.
ATTACK_SEQUENCES_FILE = MODEL_DIR / "attack_evaluation_sequences.npz"

np.savez_compressed(
    ATTACK_SEQUENCES_FILE,
    X_attack_seq=attack_sequences,
    attack_end_indices=attack_end_indices,
)

print("ATTACK SEQUENCE PREPARATION COMPLETED")
print("=" * 60)
print(f"Attack observations: {len(attack_df):,}")
print(f"Window size: {WINDOW_SIZE}")
print(f"Attack sequences: {len(attack_sequences):,}")
print(f"Sequence shape: {attack_sequences.shape}")
print(f"First sequence endpoint: {attack_sequence_timestamps.iloc[0]}")
print(f"Last sequence endpoint: {attack_sequence_timestamps.iloc[-1]}")
print(
    "All sequence labels are Attack: "
    f"{attack_sequence_labels.eq('Attack').all()}"
)
print(
    "All scaled values are finite: "
    f"{np.isfinite(attack_sequences).all()}"
)
print(f"Saved to: {ATTACK_SEQUENCES_FILE}")